# **Collective learning project -  WikiRAG-TR dataset - e5**
# **Zainab Elfatih Mohamed MAlik - 24501096**

### **0. Prepare libraries**

In [ ]:
!pip install datasets
!pip install transformers torch datasets
!pip install tqdm
!pip install faiss-cpu
!pip install evaluate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datasets import load_dataset
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import numpy as np
import faiss
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch
from tqdm import tqdm
from evaluate import load
import random

## **1. Dataset overview and preprocess**

In [ ]:
ds = load_dataset("Metin/WikiRAG-TR") #load dataset

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
train_data = ds["train"]
data = pd.DataFrame(train_data) # convert to pandas dataframe
data

,id,question,answer,context,is_negative_response,number_of_articles,ctx_split_points,correct_intro_idx
0,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Bermuda Adaları, 1609 yılında İngiliz denizci ...","Thingspiel (çoğulu Thingspiele), 1930'larda sa...",0,6,"[374, 1916, 3860, 4309, 4775, 5443]",3
1,cb5dacdf-cb15-49c5-b0a5-8de9bb77cb48,.bm uzantısının kullanımı ne zaman başladı?,.bm uzantısı 2007 yılında kullanıma açıldı.,"Kabulgan – Türk, Altay ve Moğol mitolojisinde ...",0,2,"[266, 398]",1
2,0b827c00-160b-4a1c-9e8f-096208a73892,NKVD'nin Sovyetler Birliği içindeki siyasi bas...,"NKVD, Joseph Stalin yönetimindeki Büyük Temizl...",NKVD (Narodnıy komissariyat vnutrennnih del) (...,0,3,"[1973, 3027, 3095]",0
3,0fb1e793-3744-4b29-ba49-cdec7fb66cd2,Blohin'in Stalin rejimi sırasında infazları ge...,"Vasili Blohin, Stalin rejimi sırasında infazla...",1973 yılında imzalanan 1978 yılında değiştiril...,0,6,"[825, 968, 1319, 1532, 2303, 2884]",4
4,764a2ff6-053e-49e1-8274-a5dee96f0392,Cengel Operasyonu'nun amacı neydi?,"Cengel Operasyonu, Soğuk Savaş sırasında İngil...",Waffenamt (WaA) Alman Ordusu Silah Ajansı idi....,0,3,"[1949, 2321, 3138]",2
...,...,...,...,...,...,...,...,...
5994,e01f0903-4e8a-4f0c-9e19-54abdf262f7b,Tahrik sistemlerinde kullanılan farklı hız kon...,"Üzgünüm, bu konuda sana yardımcı olamam. Başka...","Sucuk (Azerice: sucuk, Kazakça: Шұжық , Kırgı...",1,5,"[917, 1021, 1245, 1595, 1770]",-1
5995,02d45da4-47e8-4f52-8889-26b82705629c,Stabschef rütbesi SA içinde ne kadar etkiliydi...,Sorunuzu yanıtlamak için yeterli bilgiye sahip...,"Gıda Paketi, Uzun Menzilli Devriye veya ""LRP r...",1,3,"[260, 1218, 1783]",-1
5996,8d61d4d2-1718-46f6-9193-0c4d1063656d,Brief'in dijital pazarlamada önemi nedir?,"Maalesef, bu konuda sana yardımcı olamam. Yete...",Gayrinizami harp; düzenli ve büyük birlikler y...,1,5,"[2052, 2323, 2560, 2883, 3271]",-1
5997,222a9194-9fbf-4400-9a9d-dc9515d231a2,"Hardy kivi diğer isimleriyle biliniyor, bu isi...",Bu soruyu cevaplamak için yeterli bilgiye sahi...,Kendi aralarında kendilerine Scene diyen alt k...,1,3,"[283, 617, 1031]",-1


In [ ]:
'''
the function check the dataset and take questions
that have 5 or more answers 1 true 4 wrong
for questions that have more than 5 chunk it take
1 right 4 wrong
'''
# process chunks for each question
def process_chunks(row):
    chunks = []
    # parse ctx_split_points
    ctx_split_points = [0] + json.loads(row["ctx_split_points"])

    # generate all chunks
    for i in range(len(ctx_split_points) - 1):
        start, end = ctx_split_points[i], ctx_split_points[i + 1]
        chunk = row["context"][start:end]
        chunks.append((row["id"], row["question"], chunk, i == row["correct_intro_idx"]))

    # separate correct and incorrect chunks
    correct_chunks = [chunk for chunk in chunks if chunk[3]]  # correct
    incorrect_chunks = [chunk for chunk in chunks if not chunk[3]]  # incorrect

    # include questions with 5 or more chunks
    if len(chunks) >= 5:
        if len(chunks) == 5:
            # Ensure there's exactly 1 correct chunk
            if len(correct_chunks) == 1:
                return chunks
        elif len(chunks) > 5:
            # take 1 correct and 4 random incorrect chunks
            if len(correct_chunks) >= 1 and len(incorrect_chunks) >= 4:
                selected_chunks = correct_chunks[:1] + random.sample(incorrect_chunks, 4)
                return selected_chunks
    return []

In [ ]:
# process and collect valid chunks
all_chunks = []
for _, row in data.iterrows():
    valid_chunks = process_chunks(row)
    if valid_chunks:
        all_chunks.append(valid_chunks)

# flatten the list of chunks and create dataframe
flat_chunks = [chunk for question_chunks in all_chunks for chunk in question_chunks]
chunks_df = pd.DataFrame(flat_chunks, columns=["id", "question", "chunk", "is_correct"])

In [ ]:
# select 1000 questions
valid_question_ids = chunks_df["id"].unique()[:100]
final_chunks_df = chunks_df[chunks_df["id"].isin(valid_question_ids)].reset_index(drop=True)
final_chunks_df

,id,question,chunk,is_correct
0,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Bermuda, tam adıyla Bermuda Adaları (diğer adı...",True
1,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,TARDIS (Time And Relative Dimension In Space) ...,False
2,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Ekonomi tarihi veya iktisat tarihi, geçmişte y...",False
3,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Yoshiwara (Japonca: 吉原), Japonya'da Edo'da bul...",False
4,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Thingspiel (çoğulu Thingspiele), 1930'larda sa...",False
...,...,...,...,...
495,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Edmund Veesenmayer (d. 12 Kasım 1904, Bad Kiss...",False
496,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Mutajen, biyolojide canlıların DNA ya da RNA g...",False
497,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Karbon döngüsünün jeolojik bölümü, küresel kar...",True
498,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Richard Robert Ernst (d. 14 Ağustos 1933, Wint...",False


## **2. Generate embedding using multilingual-e5-large-instruct**

In [ ]:
model_name = "intfloat/multilingual-e5-large-instruct"
#"jinaai/jina-embeddings-v3"

# load tokenizer and model
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print("Model and tokenizer loaded successfully!")

# move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Loading model and tokenizer...
Model and tokenizer loaded successfully!


In [ ]:
'''
The `generate_embeddings` function turns a list of text chunks into numeric vectors (embeddings)
using a tokenizer and e5 model. It processes the text in small groups (batches) to handle large
amounts of data easily. For each group, it prepares the text, sends it through the model, and
collects the output as embeddings.
'''
def generate_embeddings(chunks, tokenizer, model, batch_size=16):
    model.eval()
    embeddings = []
    num_batches = (len(chunks) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks in batches", total=num_batches):
            batch = chunks[i:i + batch_size]

            # tokenize the batch
            inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")

            # generate embeddings using the model
            outputs = model(**inputs)

            # use the last hidden state (or other layers, depending on model specifics)
            batch_embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings.append(batch_embeddings)

    # concatenate all embeddings
    embeddings = torch.cat(embeddings, dim=0)
    return embeddings

In [ ]:
# extract chunks for embedding
chunks = final_chunks_df["chunk"].tolist()
print(f"Total chunks to embed: {len(chunks)}")

Total chunks to embed: 500


In [ ]:
print("Generating embeddings...")
chunk_embeddings = generate_embeddings(chunks, tokenizer, model)
print(f"Embeddings generated: {chunk_embeddings.shape}")

Generating embeddings...


Embedding chunks in batches: 100%|██████████| 32/32 [12:14<00:00, 22.97s/it]

Embeddings generated: torch.Size([500, 1024])


In [ ]:
# convert embeddings to numpy

chunk_embeddings_np = chunk_embeddings.cpu().numpy()

# create dataframe
embeddings_df = final_chunks_df.copy()
embeddings_df["embedding"] = list(chunk_embeddings_np)  # add embeddings as a column
embeddings_df
#embeddings_df.to_pickle("chunk_embeddings.pkl")
#print("Embeddings and metadata saved!")

,id,question,chunk,is_correct,embedding
0,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Bermuda, tam adıyla Bermuda Adaları (diğer adı...",True,"[0.88241446, 0.34341228, -0.8748609, -2.015724..."
1,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,TARDIS (Time And Relative Dimension In Space) ...,False,"[0.81970173, 0.27505437, 0.19318302, -1.467915..."
2,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Ekonomi tarihi veya iktisat tarihi, geçmişte y...",False,"[0.31600657, 0.086823106, -0.42481276, -1.1809..."
3,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Yoshiwara (Japonca: 吉原), Japonya'da Edo'da bul...",False,"[0.28770986, 0.35837758, -1.0205408, -1.440830..."
4,fdb9e733-8b3f-430e-93d4-72c563f2d00c,Bermuda Adaları'nın Birleşik Krallık'a bağlı b...,"Thingspiel (çoğulu Thingspiele), 1930'larda sa...",False,"[0.21150964, 0.9428407, -0.9870129, -0.7547478..."
...,...,...,...,...,...
495,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Edmund Veesenmayer (d. 12 Kasım 1904, Bad Kiss...",False,"[0.09107962, -0.026543343, 0.04337078, -1.3356..."
496,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Mutajen, biyolojide canlıların DNA ya da RNA g...",False,"[1.35147, 0.07230453, 0.6058244, -1.5606025, 0..."
497,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Karbon döngüsünün jeolojik bölümü, küresel kar...",True,"[0.8029852, -0.13503811, -0.7630328, -1.622635..."
498,c60f63c5-2ded-4efd-bbc3-a92c56d6a6d9,Jeosferde depolanan karbonun büyük çoğunluğu h...,"Richard Robert Ernst (d. 14 Ağustos 1933, Wint...",False,"[0.5393561, -0.08631764, 0.50687766, -2.102073..."


In [ ]:
# convert embeddings to NumPy
embedding_dim = chunk_embeddings_np.shape[1]  # Get the dimension of embeddings
print(f"Embedding dimension: {embedding_dim}")

# initialize a FAISS index for L2 (Euclidean) distance
index = faiss.IndexFlatL2(embedding_dim)

# add embeddings to the FAISS index
print("Adding embeddings to FAISS index...")
index.add(chunk_embeddings_np)
print(f"FAISS index size: {index.ntotal}")

Embedding dimension: 1024
Adding embeddings to FAISS index...
FAISS index size: 500


In [ ]:
# Save FAISS index to disk
#faiss.write_index(index, "/content/drive/My Drive/Colab Notebooks/chunk_embeddings_index5k-e5.faiss")
#print("FAISS index saved to 'chunk_embeddings_index.faiss'.")
# Save metadata (e.g., questions, chunks, is_correct) to disk
#embeddings_df.to_csv("/content/drive/My Drive/Colab Notebooks/chunk_metadata5k-e5.csv", index=False)
#print("Metadata saved to 'chunk_metadata.csv'.")


## **3. Retrieval**

In [ ]:
'''
The `retrieve_similar_chunks` function finds the most relevant text chunks for a given query.
It first converts the query into an embedding using a tokenizer and model.
Then, it compares this query embedding with pre-stored embeddings in an index to find
the top `k` most similar chunks based on distance. Finally, it retrieves metadata
(like the chunk text) for these matches and returns them along with their similarity scores.
'''
def retrieve_similar_chunks(query, tokenizer, model, index, metadata_df, top_k=5):
    # tokenize and encode the query
    inputs = tokenizer(query, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        query_embedding = model(**inputs).last_hidden_state.mean(dim=1).numpy()

    # search for top-k similar chunks, "distance metric"
    distances, indices = index.search(query_embedding, top_k)

    # retrieve corresponding metadata
    results = metadata_df.iloc[indices[0]]
    return results, distances[0]

In [ ]:
# test retrieval function
sample_query = "Bermuda Adaları hakkında bilgi verin."
results, distances = retrieve_similar_chunks(
    sample_query, tokenizer, model, index, embeddings_df, top_k=5)

print("Top-5 Retrieved Chunks:")
for i, (chunk, dist) in enumerate(zip(results["chunk"], distances)):
    print(f"{i + 1}. Distance: {dist:.4f}, Chunk: {chunk}")

Top-5 Retrieved Chunks:
1. Distance: 265.9328, Chunk: Sinop Tersanesi, Osmanlı İmparatorluğu'nun Karadeniz'deki ana tersanesi.

2. Distance: 295.1357, Chunk: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

3. Distance: 297.3138, Chunk: Ford Prefect, Otostopçunun Galaksi Rehberi serisinin baş karakterlerinden biridir.

4. Distance: 301.1673, Chunk: Hidrotermal baca, jeotermal ısıya sahip suyun salındığı bir deniz tabağı yarığıdır.

5. Distance: 308.5385, Chunk: Hawaii dini, Yerli Hawaililerin yerli dini inançlarını ve uygulamalarını kapsar.



## **4. Retrieval evaluation**


In [ ]:
'''
The `evaluate_retrieval_with_samples` function assesses the performance of a retrieval system
by testing it on a dataset of questions and their corresponding correct chunks.
For each question, it retrieves the top `k` most similar chunks and checks if the correct chunk
appears as the top result (top-1) or within the top five results (top-5). The function tracks
the number of correct top-1 and top-5 results, storing examples of correct and incorrect retrievals.
At the end, it calculates the accuracy for both top-1 and top-5 results and returns these accuracy
metrics along with detailed samples of the retrieval outcomes.
'''
def evaluate_retrieval_with_samples(metadata_df, tokenizer, model, index, top_k=5):
    correct_top1 = 0
    correct_top5 = 0
    samples = {
        "top1_correct": [],
        "top1_incorrect": [],
        "top5_incorrect": [],
        "all_retrieved": [],  # store all retrievals for each query
    }

    for _, row in metadata_df.iterrows():
        query = row["question"]
        correct_chunk = row["chunk"]

        # retrieve results
        results, _ = retrieve_similar_chunks(query, tokenizer, model, index, metadata_df, top_k=top_k)
        retrieved_chunks = results["chunk"].tolist()

        # store all retrieved results for this query
        samples["all_retrieved"].append({
            "query": query,
            "retrieved_chunks": retrieved_chunks,
            "correct_chunk": correct_chunk,
        })

        # check if the correct chunk is in the top-1 or top-5
        is_top1_correct = correct_chunk == retrieved_chunks[0]
        is_top5_correct = correct_chunk in retrieved_chunks

        # update accuracies and samples
        if is_top1_correct:
            correct_top1 += 1
            samples["top1_correct"].append({
                "query": query,
                "retrieved_chunk": retrieved_chunks[0],
                "correct_chunk": correct_chunk,
            })
        else:
            samples["top1_incorrect"].append({
                "query": query,
                "retrieved_chunk": retrieved_chunks[0],
                "correct_chunk": correct_chunk,
            })

        if not is_top5_correct:
            samples["top5_incorrect"].append({
                "query": query,
                "retrieved_chunks": retrieved_chunks,
                "correct_chunk": correct_chunk,
            })

        if is_top5_correct:
            correct_top5 += 1

    # calculate accuracies
    total_questions = len(metadata_df)
    top1_accuracy = correct_top1 / total_questions
    top5_accuracy = correct_top5 / total_questions

    # convert samples to DataFrames
    samples_dfs = {key: pd.DataFrame(value) for key, value in samples.items()}

    return top1_accuracy, top5_accuracy, samples_dfs

In [ ]:
# evaluate retrieval and get results
top1_accuracy, top5_accuracy, samples = evaluate_retrieval_with_samples(
    metadata_df=embeddings_df,
    tokenizer=tokenizer,
    model=model,
    index=index,
    top_k=5)

print(f"Top-1 Accuracy: {top1_accuracy:.2%}")
print(f"Top-5 Accuracy: {top5_accuracy:.2%}")

Top-1 Accuracy: 7.20%
Top-5 Accuracy: 10.80%


In [ ]:
for key, df in samples.items():
    print(f"{key}: {df.shape}")

top1_correct: (36, 3)
top1_incorrect: (464, 3)
top5_incorrect: (446, 3)
all_retrieved: (500, 3)


In [ ]:
# Check if 'top1_incorrect' exists in samples_fine
#if "top1_incorrect" in samples:
 #   incorrect_top1_indices = samples["top1_incorrect"].index.tolist()
  #  print("Indices of Incorrect Top-1 Retrievals:")
   # print(incorrect_top1_indices)
#else:
 #   print("The 'top1_incorrect' category is missing in samples_fine.")

In [ ]:
"""
Display retrieval results for specific indices.
- samples_fine: The dictionary containing DataFrames for retrieval results.
- indices: List of specific indices to display.
"""
def display_specific_indices(samples_fine, indices):
    if "all_retrieved" not in samples_fine:
        print("The 'all_retrieved' category is missing in samples_fine.")
        return

    all_retrieved = samples_fine["all_retrieved"]

    # ensure indices are within bounds
    if max(indices) >= len(all_retrieved) or min(indices) < 0:
        print("Error: Some indices are out of range.")
        return

    print("Retrieval Results for Specific Indices:")
    print("=" * 80)

    for idx in indices:
        row = all_retrieved.iloc[idx]
        print(f"Query: {row['query']}")
        print(f"Correct Answer: {row['correct_chunk']}")
        print(f"Retrieved Top-1: {row['retrieved_chunks'][0]}")
        print("Retrieved Top-5:")
        for i, chunk in enumerate(row['retrieved_chunks'][:5], start=1):
            print(f"  {i}. {chunk}")
        print("-" * 80)

In [ ]:
specific_indices = [0, 44, 88]  # replace with the indices you're interested in
display_specific_indices(samples, specific_indices)

Retrieval Results for Specific Indices:
Query: Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Correct Answer: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

Retrieved Top-1: Sinop Tersanesi, Osmanlı İmparatorluğu'nun Karadeniz'deki ana tersanesi.

Retrieved Top-5:
  1. Sinop Tersanesi, Osmanlı İmparatorluğu'nun Karadeniz'deki ana tersanesi.

  2. Ege Seferi, Kaptan-ı derya Hamza Bey komutasındaki Osmanlı donanmasının İstanbul'un Fethinden (1453) sonra Ege Denizi'nde hâkimiyetini yerleştirme hedefine yönelik seferlerinden ilki.

  3. Ege Seferi, K

## **5. Creating classifier**

* **Prepare the Dataset**: Create a copy of the DataFrame, convert the `is_correct` column to labels, and split the data into training (80%) and validation (20%) sets.

* **Tokenize Texts**: Define a function to tokenize the text data with truncation and padding, then apply it to the training and validation sets.

* **Create PyTorch Datasets**: Define a custom dataset class to handle the encodings and labels, and initialize training and validation datasets.

* **Initialize the Model**: Load a pretrained sequence classification model and transfer it to the appropriate device.

* **Define Metrics Function**: Set up an accuracy metric and create a function to compute it from predictions and labels.

* **Set Up the Trainer**: Specify training arguments (like output directory, evaluation strategy, and batch sizes) and create a trainer instance with the model and datasets.

* **Train the Model**: Call the `train` method to fine-tune the model on the training dataset.

* **Save the Model**: After training, save the fine-tuned model and tokenizer to a specified directory, and print a completion message.

In [ ]:
# prepare the dataset for fine-tuning
fine_tune_data = final_chunks_df.copy()
fine_tune_data["label"] = fine_tune_data["is_correct"].astype(int)

# split into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    fine_tune_data["chunk"].tolist(),
    fine_tune_data["label"].tolist(),
    test_size=0.2,
    random_state=42
)

# tokenize the dataset
def tokenize_function(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=128)

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

# convert to PyTorch datasets
class RetrievalDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = RetrievalDataset(train_encodings, train_labels)
val_dataset = RetrievalDataset(val_encodings, val_labels)

In [ ]:
#train_dataset

In [ ]:
# initialize the model for fine-tuning
fine_tune_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
fine_tune_model.to(device)

# define compute metrics function
metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), axis=1)
    accuracy = (predictions == labels).float().mean().item()
    return {"accuracy": accuracy}

# set up Trainer
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=5,
    report_to="tensorboard",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=fine_tune_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# train the model
trainer.train()

# save the fine-tuned model
fine_tune_model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")
print("Fine-tuning complete and model saved!")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at intfloat/multilingual-e5-large-instruct and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.510500,0.548055,0.780000
2,0.420600,0.560394,0.780000
3,0.586100,0.535700,0.780000
4,0.405900,0.554790,0.780000
5,0.384300,0.560154,0.780000


Fine-tuning complete and model saved!


## **5.1 Generate embeddings for classifier**

In [ ]:
def generate_fine_tuned_embeddings(chunks, tokenizer, model, batch_size=16):
    model.eval()
    embeddings = []
    num_batches = (len(chunks) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in tqdm(range(0, len(chunks), batch_size), desc="Generating fine-tuned embeddings", total=num_batches):
            batch = chunks[i:i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
            outputs = model(**inputs, output_hidden_states=True).hidden_states[-1]
            batch_embeddings = outputs.mean(dim=1)
            embeddings.append(batch_embeddings)

    embeddings = torch.cat(embeddings, dim=0)
    return embeddings

fine_tuned_embeddings = generate_fine_tuned_embeddings(fine_tune_data["chunk"].tolist(), tokenizer, fine_tune_model)
fine_tuned_embeddings_np = fine_tuned_embeddings.cpu().numpy()

Generating fine-tuned embeddings: 100%|██████████| 32/32 [10:32<00:00, 19.76s/it]


## **5.2. Retrieval an evaluation**

In [ ]:
# define retrieval and evaluation functions
def retrieve_with_classifier(query, chunks_df, tokenizer, model, top_k=5):
    scores = []
    for _, row in chunks_df.iterrows():
        question, chunk = row["question"], row["chunk"]
        inputs = tokenizer(question, chunk, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            score = torch.softmax(logits, dim=-1)[0][1].item()
            scores.append((chunk, score))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:top_k]

def evaluate_classifier(chunks_df, tokenizer, model, top_k=5):
    correct_top1 = 0
    correct_top5 = 0
    samples = {"top1_correct": [], "top1_incorrect": [], "top5_incorrect": [], "all_retrieved": []}

    for question_id in chunks_df["id"].unique():
        question_df = chunks_df[chunks_df["id"] == question_id]
        query = question_df.iloc[0]["question"]
        correct_chunk = question_df[question_df["is_correct"] == 1]["chunk"].iloc[0]
        retrieved_chunks = retrieve_with_classifier(query, question_df, tokenizer, model, top_k=top_k)
        samples["all_retrieved"].append({
            "query": query,
            "retrieved_chunks": [chunk for chunk, _ in retrieved_chunks],
            "retrieved_scores": [score for _, score in retrieved_chunks],
            "correct_chunk": correct_chunk,
        })
        is_top1_correct = correct_chunk == retrieved_chunks[0][0]
        is_top5_correct = correct_chunk in [chunk for chunk, _ in retrieved_chunks]

        if is_top1_correct:
            correct_top1 += 1
            samples["top1_correct"].append({"query": query, "retrieved_chunk": retrieved_chunks[0][0], "correct_chunk": correct_chunk})
        else:
            samples["top1_incorrect"].append({"query": query, "retrieved_chunk": retrieved_chunks[0][0], "correct_chunk": correct_chunk})

        if not is_top5_correct:
            samples["top5_incorrect"].append({"query": query, "retrieved_chunks": [chunk for chunk, _ in retrieved_chunks], "correct_chunk": correct_chunk})
        if is_top5_correct:
            correct_top5 += 1

    total_questions = chunks_df["id"].nunique()
    top1_accuracy = correct_top1 / total_questions
    top5_accuracy = correct_top5 / total_questions
    samples_dfs = {key: pd.DataFrame(value) for key, value in samples.items()}
    return top1_accuracy, top5_accuracy, samples_dfs

# evaluate the fine-tuned model
#classifier_top1, classifier_top5, samples_fine = evaluate_classifier(chunks_df=fine_tune_data, tokenizer=tokenizer, model=fine_tune_model)
#print(f"Classifier Top-1 Accuracy: {classifier_top1:.2%}")
#print(f"Classifier Top-5 Accuracy: {classifier_top5:.2%}")

In [ ]:
"""
Display retrieval results for specific queries or the first 'limit' queries.
- queries: List of specific queries to display. If None, display the first 'limit' queries.S
- limit: Number of queries to display (only used if queries is None).
"""
def display_retrieval_results(samples_fine, queries=None, limit=5):
    if "all_retrieved" not in samples_fine:
        print("The 'all_retrieved' category is missing in samples_fine.")
        return

    print("Retrieval Results:")
    print("=" * 80)
    all_retrieved = samples_fine["all_retrieved"]

    # filter rows based on specific queries if provided
    if queries:
        filtered_rows = all_retrieved[all_retrieved['query'].isin(queries)]
    else:
        filtered_rows = all_retrieved.head(limit)

    if filtered_rows.empty:
        print("No matching queries found.")
        return

    # display results
    for _, row in filtered_rows.iterrows():
        print(f"Query: {row['query']}")
        print(f"Correct Answer: {row['correct_chunk']}")
        print(f"Retrieved Top-1: {row['retrieved_chunks'][0]}")
        print("Retrieved Top-5:")
        for i, chunk in enumerate(row['retrieved_chunks'][:5], start=1):
            print(f"  {i}. {chunk}")
        print("-" * 80)


In [ ]:
specific_indices = [0, 44,88]
display_retrieval_results(samples_fine, queries=samples_fine["all_retrieved"].iloc[specific_indices]["query"].tolist())

Retrieval Results:
Query: Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Correct Answer: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

Retrieved Top-1: TARDIS (Time And Relative Dimension In Space) (Türkçe: Uzaydaki zaman ve izafi boyut), İngiliz bilimkurgu dizisi Doctor Who'daki uzay ve zamanda herhangi bir yere gidebilen, Gallifrey adlı bir gezegende yaşayan Zaman Lordları tarafından yaratılan bir zaman makinesidir. Bazen adı "The Blue Box" yani "Mavi Kulübe" olarak da geçer.
Asıl şekli olmasa da dışarıdan 1963 yılı Londrasından kalma bir poli

## **6. Classifier Fine Tuning**

In [ ]:
# prepare the dataset for fine-tuning
#fine_tune_data2 = final_chunks_df.copy()
#fine_tune_data2["label"] = fine_tune_data["is_correct"].astype(int)

# split into training and validation sets
#train_texts2, val_texts2, train_labels2, val_labels2 = train_test_split(
 #   fine_tune_data["chunk"].tolist(),
  #  fine_tune_data["label"].tolist(),
   #test_size=0.2,
    #random_state=42)

#train_encodings2 = tokenize_function(train_texts2)
#val_encodings2 = tokenize_function(val_texts2)

#train_dataset2 = RetrievalDataset(train_encodings2, train_labels2)
#val_dataset2 = RetrievalDataset(val_encodings2, val_labels2)

In [ ]:
# set up Trainer
training_args2 = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    report_to="tensorboard",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer2 = Trainer(
    model=fine_tune_model,
    args=training_args2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# train the model
trainer2.train()

# save the fine-tuned model
fine_tune_model.save_pretrained("./fine_tuned_model2")
tokenizer.save_pretrained("./fine_tuned_model2")
print("Fine-tuning complete and model2 saved!")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.524000,0.540553,0.780000
2,0.490400,0.552938,0.780000
3,0.488100,0.618963,0.780000
4,0.456900,0.535956,0.770000
5,0.388500,0.691875,0.780000
6,0.490100,0.538747,0.780000
7,0.352100,0.696465,0.760000
8,0.272400,0.813038,0.740000
9,0.233100,0.813468,0.710000
10,0.196400,0.769232,0.770000


Fine-tuning complete and model2 saved!


In [ ]:
#fine_tuned_embeddings2 = generate_fine_tuned_embeddings(fine_tune_data["chunk"].tolist(), tokenizer, fine_tune_model)
#fine_tuned_embeddings_np2 = fine_tuned_embeddings2.cpu().numpy()

In [ ]:
# evaluate the fine-tuned model
fine_top1, fine_top5, samples_fine2 = evaluate_classifier(fine_tune_data, tokenizer, fine_tune_model)
print(f"Classifier Top-1 Accuracy: {fine_top1:.2%}")
print(f"Classifier Top-5 Accuracy: {fine_top5:.2%}")

Classifier Top-1 Accuracy: 49.00%
Classifier Top-5 Accuracy: 100.00%


In [ ]:
specific_indices = [0, 44,88]
display_retrieval_results(samples_fine2, queries=samples_fine2["all_retrieved"].iloc[specific_indices]["query"].tolist())

Retrieval Results:
Query: Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Correct Answer: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

Retrieved Top-1: American Horror Story: Asylum (Türkçesi: Amerikan Korku Hikayesi: Akıl Hastanesi), FX tarafından yayınlanan korku-gerilim dizisi American Horror Story'nin ikinci sezonudur. 17 Ekim 2012 - 23 Ocak 2013 tarihleri arasında yayınlanmıştır. İkinci sezonda konu gidişatı ve karakterler ilk sezondan bağımsız olduğu için, dizi American Horror Story başlığı altında bir antoloji serisi olmuştur.
Sezon, 1964

### **fine 2**

In [ ]:
# set up Trainer
training_args3 = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    report_to="tensorboard",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer3 = Trainer(
    model=fine_tune_model,
    args=training_args3,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# train the model
trainer3.train()

# save the fine-tuned model
fine_tune_model.save_pretrained("./fine_tuned_model3")
tokenizer.save_pretrained("./fine_tuned_model3")
print("Fine-tuning complete and model3 saved!")

Epoch,Training Loss,Validation Loss,Accuracy
1,1.483200,0.537673,0.780000
2,0.520100,0.527196,0.780000
3,0.595700,0.539046,0.780000
4,0.509200,0.531265,0.780000
5,0.513100,0.536678,0.780000
6,0.519100,0.531693,0.780000
7,0.496600,0.543729,0.780000
8,0.529800,0.526908,0.780000
9,0.493000,0.534524,0.780000
10,0.500400,0.528806,0.780000


Fine-tuning complete and model3 saved!


In [ ]:
# evaluate the fine-tuned model
fine3_top1, fine3_top5, samples_fine3 = evaluate_classifier(fine_tune_data, tokenizer, fine_tune_model)
print(f"Classifier Top-1 Accuracy: {fine3_top1:.2%}")
print(f"Classifier Top-5 Accuracy: {fine3_top5:.2%}")

Classifier Top-1 Accuracy: 29.00%
Classifier Top-5 Accuracy: 100.00%


In [ ]:
specific_indices = [0, 44,88]
display_retrieval_results(samples_fine3, queries=samples_fine3["all_retrieved"].iloc[specific_indices]["query"].tolist())

Retrieval Results:
Query: Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Correct Answer: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

Retrieved Top-1: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. 

## **fine 3**

In [ ]:
# set up Trainer
training_args4 = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=1,
    report_to="tensorboard",
    learning_rate=5e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=15,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer4 = Trainer(
    model=fine_tune_model,
    args=training_args4,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# train the model
trainer4.train()

# save the fine-tuned model
fine_tune_model.save_pretrained("./fine_tuned_model4")
tokenizer.save_pretrained("./fine_tuned_model4")
print("Fine-tuning complete and model4 saved!")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.571600,0.497726,0.780000
2,0.577200,0.487879,0.810000
3,0.573900,0.528202,0.810000
4,0.286400,0.602852,0.810000
5,0.196000,0.582200,0.690000
6,0.094900,0.666664,0.790000
7,0.164000,0.889323,0.800000
8,0.011100,1.146691,0.790000
9,0.274000,1.083179,0.760000
10,0.072900,1.214690,0.790000


Epoch,Training Loss,Validation Loss,Accuracy
1,0.571600,0.497726,0.780000
2,0.577200,0.487879,0.810000
3,0.573900,0.528202,0.810000
4,0.286400,0.602852,0.810000
5,0.196000,0.582200,0.690000
6,0.094900,0.666664,0.790000
7,0.164000,0.889323,0.800000
8,0.011100,1.146691,0.790000
9,0.274000,1.083179,0.760000
10,0.072900,1.214690,0.790000


Fine-tuning complete and model4 saved!


In [ ]:
# evaluate the fine-tuned model
fine4_top1, fine4_top5, samples_fine4 = evaluate_classifier(fine_tune_data, tokenizer, fine_tune_model)
print(f"Classifier Top-1 Accuracy: {fine4_top1:.2%}")
print(f"Classifier Top-5 Accuracy: {fine4_top5:.2%}")

Classifier Top-1 Accuracy: 75.00%
Classifier Top-5 Accuracy: 100.00%


In [ ]:
specific_indices = [0, 44,88]
display_retrieval_results(samples_fine4, queries=samples_fine4["all_retrieved"].iloc[specific_indices]["query"].tolist())

Retrieval Results:
Query: Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Correct Answer: Bermuda, tam adıyla Bermuda Adaları (diğer adıyla Somers Adaları), Atlas Okyanusu'nda, ABD'nin doğu (Kuzey Carolina eyaletindeki Hatteras Burnu'nun yaklaşık 900 km doğusunda) ve Karayipler'in kuzey açıklarında bir takımadadır. Birleşik Krallık'ın denizaşırı topraklarından biridir. Ana ada olan Bermuda adası dâhil yedi ana ada ile 150 küçük ada ve kayalıktan oluşur.
Toplam yüzölçümü 53,3 km²'dir. Başkenti Hamilton'dur, nüfus 65.000 civarındadır.

Retrieved Top-1: Ekonomi tarihi veya iktisat tarihi, geçmişte yaşanan ekonomik olguların nasıl geliştiği konusunda çalışan bilim dalı. Ekonomi tarihinin analizi; tarihsel durumlara ekonomik teorilerin uygulanması, tarihsel yöntemler ve istatistiksel yöntemlerin bir kombinasyonu kullanılarak ele alınır. Bu başlık, sosyal tarihin örtüşen bazı alanları (demografik tarih gibi) ve işletme tarihi konularını içerir. Eko